In [1]:
import akshare as ak
import altair as alt
import numpy as np
import pandas as pd

## Get net asset value

In [2]:
df = ak.fund_open_fund_info_em(symbol="110020", indicator="单位净值走势")

In [3]:
df.head()

,净值日期,单位净值,日增长率
0,2009-08-26,1.000,0.0000
1,2009-08-28,0.998,-0.2000
2,2009-09-04,1.005,0.7014
3,2009-09-11,1.016,1.0945
4,2009-09-15,1.019,0.2953


In [4]:
len(df)

4138

In [5]:
df["净值日期"] = pd.to_datetime(df["净值日期"])

In [6]:
del df["日增长率"]

## Plot net asset value

In [7]:
alt.Chart(df).mark_line().encode(
    x="净值日期",
    y="单位净值",
).properties(
    title="易方达沪深 300 基金净值走势图",
    width=700,
    height=500,
)

alt.Chart(...)

## Define some types and helper functions

In [8]:
Point = tuple[float, float]

In [9]:
def triangle_area(p1: Point, p2: Point, p3: Point):
    return 0.5 * abs(
        (p2[0] - p1[0]) * (p3[1] - p1[1]) - (p2[1] - p1[1]) * (p3[0] - p1[0])
    )

## Implement LTTB Algorithm

In [10]:
def lttb(points: list[Point], target_node_cnt: int):
    if len(points) <= 2:
        return points

    ret: list[Point] = [points[0]]
    bucket_size = (len(points) - 2) / (target_node_cnt - 2)

    for i in range(target_node_cnt - 2):
        # Find the average of the next bucket
        next_bucket_start = int((i + 1) * bucket_size) + 1
        next_bucket_end = min(int((i + 2) * bucket_size + 1), len(points))
        next_bucket_size = next_bucket_end - next_bucket_start
        avg_x, avg_y = 0.0, 0.0
        for ni in range(next_bucket_start, next_bucket_end):
            avg_x += points[ni][0]
            avg_y += points[ni][1]
        avg_x, avg_y = avg_x / next_bucket_size, avg_y / next_bucket_size

        # Find best node in current bucket
        cur_bucket_start = int(i * bucket_size) + 1
        cur_bucket_end = int((i + 1) * bucket_size) + 1
        max_area, best_choice = 0, 0
        for ni in range(cur_bucket_start, cur_bucket_end):
            area = triangle_area(ret[-1], points[ni], (avg_x, avg_y))
            if area > max_area:
                max_area, best_choice = area, ni
        ret.append(points[best_choice])

    ret.append(points[-1])

    return ret

In [11]:
indices = list(range(len(df)))
nav = df["单位净值"].to_list()

points = [(x, y) for x, y in zip(indices, nav)]

In [12]:
TARGET_NODE_CNT = 42

In [13]:
lttb_points = lttb(points, TARGET_NODE_CNT)
lttb_indices = [x[0] for x in lttb_points]
len(lttb_points)

42

In [14]:
df["type"] = "ORIGIN"

In [15]:
lttb_df = df.iloc[lttb_indices].copy()
lttb_df.head()

,净值日期,单位净值,type
0,2009-08-26,1.000,ORIGIN
57,2009-12-07,1.100,ORIGIN
196,2010-07-05,0.767,ORIGIN
278,2010-11-08,1.067,ORIGIN
330,2011-01-20,0.893,ORIGIN


In [16]:
lttb_df["type"] = f"LTTB (M={TARGET_NODE_CNT})"

In [17]:
plot_df = pd.concat([df, lttb_df], axis=0)
plot_df.head()

,净值日期,单位净值,type
0,2009-08-26,1.000,ORIGIN
1,2009-08-28,0.998,ORIGIN
2,2009-09-04,1.005,ORIGIN
3,2009-09-11,1.016,ORIGIN
4,2009-09-15,1.019,ORIGIN


In [18]:
alt.Chart(plot_df).mark_line().encode(
    x="净值日期",
    y="单位净值",
    color="type:N"
).properties(
    width=700,
    height=500,
)

alt.Chart(...)

## MinMaxLTTB

In [19]:
def pre_selection(
    points: list[Point],
    target_node_cnt: int,
    ratio: int,
):
    if len(points) <= 2:
        return points

    if ratio < 2 or ratio % 2 == 1:
        raise ValueError(f"Invalid ratio: {ratio}")

    ret: list[Point] = [points[0]]

    num_partitions = (target_node_cnt * ratio - 2) // 2
    if num_partitions > len(points) - 2:
        raise ValueError(
            f"Too many partitions ({num_partitions}) for nodes ({len(points) - 2})"
        )

    partition_size = (len(points) - 2) / num_partitions

    for i in range(num_partitions):
        range_start = int(i * partition_size) + 1
        range_end = min(int((i + 1) * partition_size) + 1, len(points) - 1)
        min_val, min_idx = float("inf"), -1
        max_val, max_idx = -float("inf"), -1

        for j in range(range_start, range_end):
            if points[j][1] < min_val:
                min_val, min_idx = points[j][1], j

            if points[j][1] > max_val:
                max_val, max_idx = points[j][1], j

        if min_idx > max_idx:
            max_idx, min_idx = min_idx, max_idx

        ret.append(points[min_idx])
        ret.append(points[max_idx])

    ret.append(points[-1])

    assert len(ret) == target_node_cnt * ratio

    return ret

In [20]:
def minmax_lttb(
    points: list[Point],
    target_node_cnt: int,
    ratio: int
):
    pre_selected_nodes = pre_selection(points, TARGET_NODE_CNT, ratio=ratio)
    return lttb(pre_selected_nodes, TARGET_NODE_CNT)

In [21]:
def make_df_with_type(df, points, typename):
    indices = [x[0] for x in points]
    ret = df.iloc[indices].copy()
    ret["type"] = typename
    return ret

In [22]:
minmax_lttb_points_ratio_2 = minmax_lttb(points, TARGET_NODE_CNT, 2)

In [23]:
minmax_lttb_points_ratio_4 = minmax_lttb(points, TARGET_NODE_CNT, 4)

In [24]:
minmax_lttb_df_ratio_2 = make_df_with_type(df, minmax_lttb_points_ratio_2, f"MinMaxLTTB (M={TARGET_NODE_CNT}, ratio=2)")

In [25]:
minmax_lttb_df_ratio_4 = make_df_with_type(df, minmax_lttb_points_ratio_4, f"MinMaxLTTB (M={TARGET_NODE_CNT}, ratio=4)")

In [26]:
plot_df = pd.concat([df, lttb_df, minmax_lttb_df_ratio_2, minmax_lttb_df_ratio_4], axis=0)
plot_df.head()

,净值日期,单位净值,type
0,2009-08-26,1.000,ORIGIN
1,2009-08-28,0.998,ORIGIN
2,2009-09-04,1.005,ORIGIN
3,2009-09-11,1.016,ORIGIN
4,2009-09-15,1.019,ORIGIN


In [27]:
alt.Chart(plot_df).mark_line().encode(
    x="净值日期",
    y="单位净值",
    color=alt.Color(
        "type:N",
        scale=alt.Scale(
            domain=[
                "ORIGIN",
                f"LTTB (M={TARGET_NODE_CNT})",
                f"MinMaxLTTB (M={TARGET_NODE_CNT}, ratio=2)",
                f"MinMaxLTTB (M={TARGET_NODE_CNT}, ratio=4)",
            ],
            range=["#9E9E9E", "#1f77b4", "#ff7f0e", "#2ca02c"],
        ),
    ),
).properties(
    width=700,
    height=500,
)

alt.Chart(...)